In [1]:
!git clone https://github.com/tiny-smart/dysample.git
!git clone https://github.com/cvlab-stonybrook/LearningToCountEverything.git
!ls dysample
!ls LearningToCountEverything

Cloning into 'dysample'...
remote: Enumerating objects: 57, done.
remote: Counting objects: 100% (57/57), done.
remote: Compressing objects: 100% (53/53), done.
remote: Total 57 (delta 22), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (57/57), 30.82 KiB | 2.80 MiB/s, done.
Resolving deltas: 100% (22/22), done.
Cloning into 'LearningToCountEverything'...
remote: Enumerating objects: 93, done.
remote: Counting objects: 100% (37/37), done.
remote: Compressing objects: 100% (10/10), done.
remote: Total 93 (delta 30), reused 27 (delta 27), pack-reused 56 (from 2)
Receiving objects: 100% (93/93), 10.51 MiB | 7.22 MiB/s, done.
Resolving deltas: 100% (44/44), done.
complexity.jpg	dysample.py  LICENSE  README.md
data			   LICENSE	      orange.jpg  train.py
demo.py			   model.py	      README.md   utils.py
learn2countEverything.png  orange_box_ex.txt  test.py


In [2]:
!cat dysample/dysample.py

import torch
import torch.nn as nn
import torch.nn.functional as F


def normal_init(module, mean=0, std=1, bias=0):
    if hasattr(module, 'weight') and module.weight is not None:
        nn.init.normal_(module.weight, mean, std)
    if hasattr(module, 'bias') and module.bias is not None:
        nn.init.constant_(module.bias, bias)


def constant_init(module, val, bias=0):
    if hasattr(module, 'weight') and module.weight is not None:
        nn.init.constant_(module.weight, val)
    if hasattr(module, 'bias') and module.bias is not None:
        nn.init.constant_(module.bias, bias)


class DySample(nn.Module):
    def __init__(self, in_channels, scale=2, style='lp', groups=4, dyscope=False):
        super().__init__()
        self.scale = scale
        self.style = style
        self.groups = groups
        assert style in ['lp', 'pl']
        if style == 'pl':
            assert in_channels >= scale ** 2 and in_channels % scale ** 2 == 0
        assert in_channels >= groups and in

In [3]:
import os
root = "LearningToCountEverything/data"   # path to where you extracted FSC-147
for d, dirs, files in os.walk(root):
    depth = d[len(root):].count(os.sep)
    if depth <= 1:
        print(d, "| subfolders:", len(dirs), "| files:", len(files))
        print("   sample:", sorted(files)[:5])

LearningToCountEverything/data | subfolders: 1 | files: 3
   sample: ['ImageClasses_FSC147.txt', 'Train_Test_Val_FSC_147.json', 'annotation_FSC147_384.json']
LearningToCountEverything/data/pretrainedModels | subfolders: 0 | files: 1
   sample: ['FamNet_Save1.pth']


In [4]:
import json, glob
for f in glob.glob(root + "/**/*.json", recursive=True):
    print(f)

LearningToCountEverything/data/Train_Test_Val_FSC_147.json
LearningToCountEverything/data/annotation_FSC147_384.json


In [7]:
import json

ann = json.load(open("LearningToCountEverything/data/annotation_FSC147_384.json"))
split = json.load(open("LearningToCountEverything/data/Train_Test_Val_FSC_147.json"))

k = list(ann.keys())[0]
print("num images:", len(ann), "| example key:", k)
print({kk: (v if kk != "points" else str(v)[:120]) for kk, v in ann[k].items()})

print("split keys:", list(split.keys()))
for s in split:
    print(s, len(split[s]), split[s][:2])

counts = [len(v["points"]) for v in ann.values()]
import numpy as np
print("count min/median/max:", min(counts), int(np.median(counts)), max(counts))

num images: 6146 | example key: 1050.jpg
{'H': 1065, 'W': 1300, 'box_examples_coordinates': [[[143, 266], [143, 317], [216, 317], [216, 266]], [[297, 66], [297, 86], [343, 86], [343, 66]], [[151, 102], [151, 139], [186, 139], [186, 102]]], 'box_examples_path': ['/nfs/bigneuron/viresh/FSC_NewDataOnly/box_examples/1050_0.jpg', '/nfs/bigneuron/viresh/FSC_NewDataOnly/box_examples/1050_1.jpg'], 'density_path': '/nfs/bigneuron/viresh/FSC_NewDataOnly/gt_density_map_adaptive_384_VarV2/1050.npy', 'density_path_fixed': '/nfs/bigneuron/viresh/FSC_NewDataOnly/gt_density_map_fixed/1050.npy', 'img_path': '/nfs/bigneuron/viresh/FSC_NewDataOnly/images_384_VarV2/1050.jpg', 'points': '[[247.53459230769232, 212.71076056338032], [247.14496153846153, 194.39414084507044], [229.89658461538463, 66.21746478873', 'r': [29, 29], 'ratio_h': 0.36056338028169016, 'ratio_w': 0.3607692307692308}
split keys: ['test', 'test_coco', 'train', 'val', 'val_coco']
test 1190 ['2.jpg', '3.jpg']
test_coco 282 ['685.jpg', '686.j

In [6]:
print(glob.glob(root + "/**/*Train_Test_Val*", recursive=True))

['LearningToCountEverything/data/Train_Test_Val_FSC_147.json']


In [8]:
!pip -q install gdown
!gdown 1ymDYrGs9DSRicfZbSCDiOu0ikGDh5k6S -O /content/fsc147_images.zip
!unzip -q /content/fsc147_images.zip -d /content/fsc147
!ls /content/fsc147

Downloading...
From (original): https://drive.google.com/uc?id=1ymDYrGs9DSRicfZbSCDiOu0ikGDh5k6S
From (redirected): https://drive.google.com/uc?id=1ymDYrGs9DSRicfZbSCDiOu0ikGDh5k6S&confirm=t&uuid=9366e403-6156-4f13-9e60-dfa87734e4d6
To: /content/fsc147_images.zip
100% 1.53G/1.53G [00:19<00:00, 77.4MB/s]
gt_density_map_adaptive_384_VarV2  images_384_VarV2


In [9]:
import os, glob
root = "/content/fsc147"
for d, dirs, files in os.walk(root):
    if d.count(os.sep) - root.count(os.sep) <= 2:
        print(d, "| subfolders:", len(dirs), "| files:", len(files))
        print("   sample:", sorted(files)[:3])

print("jpg count:", len(glob.glob(root + "/**/*.jpg", recursive=True)))
print("npy count:", len(glob.glob(root + "/**/*.npy", recursive=True)))

/content/fsc147 | subfolders: 2 | files: 0
   sample: []
/content/fsc147/images_384_VarV2 | subfolders: 0 | files: 6146
   sample: ['1050.jpg', '1053.jpg', '1061.jpg']
/content/fsc147/gt_density_map_adaptive_384_VarV2 | subfolders: 0 | files: 6146
   sample: ['1050.npy', '1053.npy', '1061.npy']
jpg count: 6146
npy count: 6146


In [10]:
import json, numpy as np
from PIL import Image

ann = json.load(open("LearningToCountEverything/data/annotation_FSC147_384.json"))
k = "1050.jpg"
img_path = glob.glob(root + f"/**/{k}", recursive=True)
den_path = glob.glob(root + f"/**/{k.replace('.jpg', '.npy')}", recursive=True)
print(img_path, den_path)

img = Image.open(img_path[0])
den = np.load(den_path[0])
print("image size (W,H):", img.size)
print("density shape:", den.shape, "| sum:", den.sum())
print("num points in annotation:", len(ann[k]["points"]))

['/content/fsc147/images_384_VarV2/1050.jpg'] ['/content/fsc147/gt_density_map_adaptive_384_VarV2/1050.npy']
image size (W,H): (469, 384)
density shape: (384, 469) | sum: 136.00002
num points in annotation: 136


In [16]:
import json, os, numpy as np, torch, cv2
from PIL import Image
from transformers import AutoModel

device = "cuda"
root = "/content/fsc147"
split = json.load(open("LearningToCountEverything/data/Train_Test_Val_FSC_147.json"))

dino = AutoModel.from_pretrained("facebook/dinov2-small").to(device).half().eval()

S = 378          # input size (27 x 14)
G = S // 14      # 27 tokens per side
O = 108          # target density size (4 x token grid)
mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
std = np.array([0.229, 0.224, 0.225], dtype=np.float32)

def load_img(name):
    im = Image.open(f"{root}/images_384_VarV2/{name}").convert("RGB").resize((S, S), Image.BICUBIC)
    x = (np.asarray(im, dtype=np.float32) / 255.0 - mean) / std
    return x.transpose(2, 0, 1)

def load_den(name):
    d = np.load(f"{root}/gt_density_map_adaptive_384_VarV2/{name.replace('.jpg', '.npy')}").astype(np.float32)
    total = float(d.sum())
    d2 = cv2.resize(d, (O, O), interpolation=cv2.INTER_AREA)
    if d2.sum() > 0:
        d2 *= total / d2.sum()
    return d2, total

os.makedirs("/content/cache", exist_ok=True)

@torch.no_grad()
def extract(split_name, bs=32):
    names = split[split_name]
    n = len(names)
    feats = np.lib.format.open_memmap(f"/content/cache/{split_name}_feat.npy", mode="w+",
                                      dtype=np.float16, shape=(n, 384, G, G))
    dens = np.zeros((n, O, O), np.float32)
    cnts = np.zeros(n, np.float32)
    for i in range(0, n, bs):
        batch = names[i:i + bs]
        x = torch.tensor(np.stack([load_img(b) for b in batch])).to(device).half()
        h = dino(pixel_values=x).last_hidden_state[:, 1:, :]            # drop CLS token
        h = h.reshape(len(batch), G, G, 384).permute(0, 3, 1, 2)
        feats[i:i + len(batch)] = h.cpu().numpy()
        for j, b in enumerate(batch):
            dens[i + j], cnts[i + j] = load_den(b)
    feats.flush()
    np.save(f"/content/cache/{split_name}_den.npy", dens)
    np.save(f"/content/cache/{split_name}_cnt.npy", cnts)
    print(split_name, n, "done")

for s in ["train", "val", "test"]:
    extract(s)

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

train 3659 done
val 1286 done
test 1190 done


In [17]:
for s in ["train", "val", "test"]:
    f = np.load(f"/content/cache/{s}_feat.npy", mmap_mode="r")
    d = np.load(f"/content/cache/{s}_den.npy")
    c = np.load(f"/content/cache/{s}_cnt.npy")
    print(s, f.shape, "feat nan:", bool(np.isnan(f[:50].astype(np.float32)).any()),
          "| den sum vs cnt (first 3):", d[:3].sum(axis=(1, 2)), c[:3])

train (3659, 384, 27, 27) feat nan: False | den sum vs cnt (first 3): [13.        8.        8.999998] [13.        8.        8.999999]
val (1286, 384, 27, 27) feat nan: False | den sum vs cnt (first 3): [13.       14.999999 18.999996] [13.       15.       18.999996]
test (1190, 384, 27, 27) feat nan: False | den sum vs cnt (first 3): [ 8.000001 10.999998  9.999998] [ 8.000001 10.999998  9.999998]


In [18]:
from google.colab import drive
drive.mount('/content/drive')
!cp -r /content/cache /content/drive/MyDrive/fsc147_cache

MessageError: Error: credential propagation was unsuccessful

In [19]:
import sys, math, torch, torch.nn as nn, torch.nn.functional as F, numpy as np
sys.path.append("dysample")
from dysample import DySample

dev = "cuda"
def load(s):
    f = torch.from_numpy(np.load(f"/content/cache/{s}_feat.npy")).to(dev)   # fp16
    d = torch.from_numpy(np.load(f"/content/cache/{s}_den.npy")).to(dev)
    c = torch.from_numpy(np.load(f"/content/cache/{s}_cnt.npy")).to(dev)
    return f, d, c
tr, va, te = load("train"), load("val"), load("test")

SCALE = 100.0

class CountNet(nn.Module):
    def __init__(self, up):
        super().__init__()
        self.proj = nn.Sequential(nn.Conv2d(384, 128, 1), nn.BatchNorm2d(128), nn.GELU())
        if up == "bilinear":
            self.up = nn.Upsample(scale_factor=4, mode="bilinear", align_corners=False)
        else:
            self.up = DySample(128, scale=4, style="lp", groups=4)
        self.head = nn.Sequential(nn.Conv2d(128, 64, 3, padding=1), nn.GELU(), nn.Conv2d(64, 1, 1))

    def forward(self, x):
        return F.relu(self.head(self.up(self.proj(x))))

@torch.no_grad()
def evaluate(m, data, bs=64):
    m.eval()
    errs = []
    for i in range(0, len(data[0]), bs):
        p = m(data[0][i:i + bs].float()).squeeze(1).sum(dim=(1, 2)) / SCALE
        errs.append(p - data[2][i:i + bs])
    e = torch.cat(errs)
    return e.abs().mean().item(), e.pow(2).mean().sqrt().item()

def run(up, seed, epochs=30, bs=32, lr=1e-3, verbose=True):
    torch.manual_seed(seed); np.random.seed(seed)
    m = CountNet(up).to(dev)
    opt = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=1e-4)
    n = len(tr[0])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs * math.ceil(n / bs))
    for ep in range(epochs):
        m.train()
        perm = torch.randperm(n, device=dev)
        for i in range(0, n, bs):
            idx = perm[i:i + bs]
            pred = m(tr[0][idx].float()).squeeze(1)
            loss = F.mse_loss(pred, tr[1][idx] * SCALE)
            opt.zero_grad(); loss.backward(); opt.step(); sched.step()
        if verbose and (ep + 1) % 2 == 0:
            mae, rmse = evaluate(m, va)
            print(f"ep {ep + 1}: val MAE={mae:.2f} RMSE={rmse:.2f}", flush=True)
    return m

# sanity reference: always predicting the median training count
med = tr[2].median()
print("val MAE of constant (median) predictor:", (va[2] - med).abs().mean().item())

m = run("bilinear", seed=0, epochs=10)

val MAE of constant (median) predictor: 48.44401168823242
ep 2: val MAE=34.23 RMSE=97.20
ep 4: val MAE=33.63 RMSE=95.37
ep 6: val MAE=33.62 RMSE=95.13
ep 8: val MAE=33.02 RMSE=94.63
ep 10: val MAE=32.77 RMSE=93.57


In [20]:
import time, pandas as pd

results = []
for up in ["bilinear", "dysample"]:
    for seed in [0, 1, 2]:
        t0 = time.time()
        m = run(up, seed=seed, epochs=30, verbose=False)
        v_mae, v_rmse = evaluate(m, va)
        t_mae, t_rmse = evaluate(m, te)
        results.append(dict(upsampler=up, seed=seed, val_mae=v_mae, val_rmse=v_rmse,
                            test_mae=t_mae, test_rmse=t_rmse, params=sum(p.numel() for p in m.up.parameters())))
        print(up, seed, f"val MAE={v_mae:.2f} test MAE={t_mae:.2f}  ({time.time() - t0:.0f}s)", flush=True)
        del m; torch.cuda.empty_cache()

df = pd.DataFrame(results)
df.to_csv("counting_results.csv", index=False)
print(df.groupby("upsampler")[["val_mae", "val_rmse", "test_mae", "test_rmse"]].agg(["mean", "std"]).round(2))
print("upsampler params:", df.groupby("upsampler").params.first().to_dict())

bilinear 0 val MAE=32.21 test MAE=26.39  (149s)
bilinear 1 val MAE=32.01 test MAE=27.17  (154s)
bilinear 2 val MAE=32.39 test MAE=27.24  (156s)


/usr/local/lib/python3.13/dist-packages/torch/functional.py:504: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/native/TensorShape.cpp:4221.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


dysample 0 val MAE=32.14 test MAE=25.63  (162s)
dysample 1 val MAE=30.79 test MAE=25.89  (162s)
dysample 2 val MAE=32.27 test MAE=26.28  (161s)
          val_mae       val_rmse       test_mae       test_rmse      
             mean   std     mean   std     mean   std      mean   std
upsampler                                                            
bilinear    32.21  0.19    95.36  0.69    26.93  0.47    127.06  0.48
dysample    31.73  0.82    95.19  0.43    25.93  0.33    127.34  1.11
upsampler params: {'bilinear': 0, 'dysample': 16512}


In [21]:
@torch.no_grad()
def predict(m, data, bs=64):
    m.eval()
    out = []
    for i in range(0, len(data[0]), bs):
        out.append(m(data[0][i:i + bs].float()).squeeze(1).sum(dim=(1, 2)) / SCALE)
    return torch.cat(out).cpu().numpy()

true = te[2].cpu().numpy()
rows = []
for up in ["bilinear", "dysample"]:
    for seed in [0, 1, 2]:
        m = run(up, seed=seed, epochs=30, verbose=False)
        pred = predict(m, te)
        rows += [(up, seed, i, true[i], pred[i]) for i in range(len(true))]
        del m; torch.cuda.empty_cache()

pdf = pd.DataFrame(rows, columns=["upsampler", "seed", "idx", "true", "pred"])
pdf["abs_err"] = (pdf.pred - pdf.true).abs()
pdf["bin"] = pd.cut(pdf.true, [0, 20, 50, 100, 10000], labels=["<=20", "21-50", "51-100", ">100"])
print(pdf.groupby(["bin", "upsampler"], observed=True).abs_err.mean().unstack().round(2))
print(pdf.groupby("bin", observed=True).idx.nunique())
pdf.to_csv("counting_per_image_test.csv", index=False)

upsampler   bilinear   dysample
bin                            
<=20        6.690000   6.010000
21-50      12.220000  10.590000
51-100     23.430000  22.490000
>100       95.120003  94.730003
bin
<=20      315
21-50     424
51-100    253
>100      198
Name: idx, dtype: int64
